In [2]:
import csv
import io
import unittest
import xml.etree.ElementTree as ET
from abc import ABC, abstractmethod
from collections import Counter
from contextlib import contextmanager

import polib
from langdetect import DetectorFactory, LangDetectException, detect

DetectorFactory.seed = 0

# Задание 1

## Часть 1

Напишите такой класс Animal, чтобы при вызове его классового метода get_highest() возвращался рост самого высокого из экземпляров

In [3]:
class Animal:
    _highest = 0

    def __init__(self, name, height):
        self.name = name
        self.height = height
        Animal._highest = max(Animal._highest, height)

    def speak(self):
        print("...")

    @classmethod
    def get_highest(cls):
        return Animal._highest

In [4]:
a1 = Animal("Alex", 200)
a2 = Animal("Melman", 400)
a3 = Animal("King Julien XIII", 40)
a4 = Animal("Maurice", 40)
assert Animal.get_highest() == 400

## Часть 2

Все же считать всех животных просто.. животными это странно. Давайте заведем каждому животному по своему классу, унаследовавшись от класса Animal, при этом сделайте так, чтобы при сравнении двух животных только для животных одного вида возвращалось True

In [5]:
class Lion(Animal):
    def __eq__(self, other):
        return type(self) is type(other)


class Lemur(Animal):
    def __eq__(self, other):
        return type(self) is type(other)


class Giraffe(Animal):
    def __eq__(self, other):
        return type(self) is type(other)

In [6]:
a1 = Lion("Alex", 200)
a2 = Giraffe("Melman", 400)
a3 = Lemur("King Julien XIII", 40)
a4 = Lemur("Maurice", 40)
assert a4 == a3
assert a1 != a2
assert a2 != a4

## Часть 3

Это все здорово, но как-то неправильно, что класс Animal имеет метод speak, а классы-потомки не определяют свою реализацию. Давайте сделаем так, чтобы все виды животных были обязаны иметь свою реализацию метода speak, но просто animal'ов уже нельзя было создать

In [7]:
class Animal(ABC):
    _highest = 0

    def __init__(self, name, height):
        self.name = name
        self.height = height
        Animal._highest = max(Animal._highest, height)

    def __eq__(self, other):
        return type(self) is type(other)

    @abstractmethod
    def speak(self):
        raise NotImplementedError

    @classmethod
    def get_highest(cls):
        return Animal._highest


class Lion(Animal):
    def speak(self):
        return "Р-р-р!"


class Lemur(Animal):
    def speak(self):
        return "Пи-пи!"


class Giraffe(Animal):
    def speak(self):
        return "Хм-м!"

In [8]:
class AnimalTestCase(unittest.TestCase):
  @contextmanager
  def assertNotRaises(self, exc_type):
    print(0)
    try:
      yield None
    except exc_type:
      raise self.failureException('{} raised'.format(exc_type.__name__))

  def test_create_animal_instance(self):
    with self.assertRaises(TypeError):
      a = Animal("", 0)

  def test_create_zebra(self):
    with self.assertRaises(TypeError):
      class Zebra(Animal):
        pass
      a = Zebra("", 0)

  def test_create_animals(self):
    with self.assertNotRaises(TypeError):
      a1 = Lion("Alex", 200)
      a2 = Giraffe("Melman", 400)
      a3 = Lemur("King Julien XIII", 40)
      a4 = Lemur("Maurice", 40)
      a1.speak()
      a2.speak()
      a3.speak()
      a4.speak()

unittest.main(argv=[''], verbosity=0, exit=False)

----------------------------------------------------------------------
Ran 3 tests in 0.000s

OK


0


Вы должны увидеть такой результат

0
----------------------------------------------------------------------
Ran 3 tests in 0.002s

OK
<unittest.main.TestProgram at 0x7f081faf2190>

# Задание 2

## Исходные данные

Представим ситуацию, что вы нашли где-то целый архив с файлами переводов и решили обработать эти файлы.  Проблема в том, что все файлы имеют разные форматы:

In [9]:
#.xliff
file1 = '''
<?xml version="1.0" encoding="UTF-8"?>
<xliff version="1.2"
  xmlns="urn:oasis:names:tc:xliff:document:1.2"
  xmlns:xsi="http://www.w3.org/2001/XMLSchema-instance"
  xsi:schemaLocation="urn:oasis:names:tc:xliff:document:1.2 xliff-core-1.2-strict.xsd">
<file original="sample.properties"
  source-language="en" target-language="de-AT"
  datatype="javapropertyresourcebundle">
<body>
   <trans-unit id="1" resname="welcome">
      <source xml:lang="en">Hooray, you're here! The day just got better - enjoy the following tips!</source>
      <target xml:lang="de-AT">Hurra, du bist hier! Der Tag ist gerettet - nutze die folgenden Tipps!</target>
   </trans-unit>
   <trans-unit id="2" resname="text_segment">
      <source xml:lang="en">A section of text like this is known as a text segment. Start rockin' your translations now!</source>
      <target xml:lang="de-AT">Eine Textpassage wie diese bezeichnen wir als Textsegment. Starte jetzt mit deinen Übersetzungen durch!</target>
   </trans-unit>
   <trans-unit id="3" resname="tab_shortcut">
      <source xml:lang="en">Arriba, Arriba! Andale, Andale! Be fast as Speedy Gonzales. Just hit TAB to save and go to the next text segment, once you're done.</source>
      <target xml:lang="de-AT">Arriba, Arriba! Andale, Andale! Sei schneller als Speedy Gonzales. Springe mit TAB ins nächste Textsegment, sobald du fertig bist. Deine Änderungen werden automatisch gespeichert.</target>
   </trans-unit>
   <trans-unit id="4" resname="love">
      <source xml:lang="en">Made with ❤</source>
      <target xml:lang="de-AT">❤</target>
      <note>lh-check { min: 10, max: 15 }</note>
   </trans-unit>
</body>
</file>
</xliff>
'''

#.po
file2 = r'''
msgid ""
msgstr ""
"Project-Id-Version: LingoHub 2.2.4\n"
"Report-Msgid-Bugs-To: \n"
"POT-Creation-Date: 2015-03-05 19:22+0200\n"
"PO-Revision-Date: 2015-04-29 10:16+0100\n"
"Last-Translator: Translator <translator@lingohub.com>\n"
"MIME-Version: 1.0\n"
"Content-Type: text/plain; charset=UTF-8\n"
"Content-Transfer-Encoding: 8bit\n"
"Plural-Forms: nplurals=2; plural=n>1;\n"

msgid "Hooray, you're here! The day just got better - enjoy the following tips!"
msgstr "Hurra, du bist hier! Der Tag ist gerettet - nutze die folgenden Tipps!"

msgid "A section of text like this is known as a text segment. Start rockin' your translations now!"
msgstr "Eine Textpassage wie diese bezeichnen wir als Textsegment. Starte jetzt mit deinen Übersetzungen durch!"

msgid "Arriba, Arriba! Andale, Andale! Be fast as Speedy Gonzales. Just hit TAB to save and go to the next text segment, once you're done."
msgstr "Arriba, Arriba! Andale, Andale! Sei schneller als Speedy Gonzales. Springe mit TAB ins nächste Textsegment, sobald du fertig bist. Deine Änderungen werden automatisch gespeichert."

msgid "Made with ❤"
msgstr "❤"
'''

#.po
file3 = r'''
msgid ""
msgstr ""
"Project-Id-Version: LingoHub 2.2.4\n"
"Report-Msgid-Bugs-To: \n"
"POT-Creation-Date: 2015-03-05 19:22+0200\n"
"PO-Revision-Date: 2015-04-29 10:16+0100\n"
"Last-Translator: Translator <translator@lingohub.com>\n"
"MIME-Version: 1.0\n"
"Content-Type: text/plain; charset=UTF-8\n"
"Content-Transfer-Encoding: 8bit\n"
"Plural-Forms: nplurals=2; plural=n>1;\n"

msgid "Hooray, you're here! The day just got better - enjoy the following tips!"
msgstr "Hurrah, tu es là. Ça fait une bonne journée - profite des renseignements suivants !"

msgid "A section of text like this is known as a text segment. Start rockin' your translations now!"
msgstr "Un tel extrait est qualifié de segment du texte chez nous. C’est parti – commence tes traductions tout de suite/maintenant !"

msgid "Arriba, Arriba! Andale, Andale! Be fast as Speedy Gonzales. Just hit TAB to save and go to the next text segment, once you're done."
msgstr "Arriba, Arriba ! Andale, Andale ! Prends Speedy Gonzales de vitesse. Appuie sur TAB pour sauvegarder et continuer avec le segment prochain dès que tu as fini."

msgid "We just like to see you happy, that's why LingoChecks automatically check translations for predetermined criteria.\n Among other checks LingoHub verifies if %s used in the original text are also present in translated texts."
msgstr "Bazinga !"

msgid "Made with ❤"
msgstr "❤"
'''

#.ts
file4 = '''
<?xml version="1.0" encoding="utf-8"?>
<!DOCTYPE TS>
<TS version="2.1" language="de_AT">
<context>
    <name>examples</name>
    <message>
        <source>Hooray, you're here! The day just got better - enjoy the following tips!</source>
        <translation>Hurra, du bist hier! Der Tag ist gerettet - nutze die folgenden Tipps!</translation>
    </message>
    <message>
        <source>A section of text like this is known as a text segment. Start rockin' your translations now!</source>
	      <translation>Eine Textpassage wie diese bezeichnen wir als Textsegment. Starte jetzt mit deinen Übersetzungen durch!</translation>
    </message>
    <message>
        <source>Arriba, Arriba! Andale, Andale! Be fast as Speedy Gonzales. Just hit TAB to save and go to the next text segment, once you're done.</source>
	      <translation>Arriba, Arriba! Andale, Andale! Sei schneller als Speedy Gonzales. Springe mit TAB ins nächste Textsegment, sobald du fertig bist. Deine Änderungen werden automatisch gespeichert.</translation>
    </message>
    <message>
        <source>Made with ❤</source>
        <translation>❤</translation>
        <comment>lh-check { min: 10, max: 15 }</comment>
    </message>
</context>
</TS>
'''

#.ts
file5 = '''
<?xml version="1.0" encoding="utf-8"?>
<!DOCTYPE TS>
<TS version="2.1" language="fr">
<context>
    <name>examples</name>
    <message>
        <source>Hooray, you're here! The day just got better - enjoy the following tips!</source>
        <translation>Hurrah, tu es là. Ça fait une bonne journée - profite des renseignements suivants !</translation>
    </message>
    <message>
        <source>A section of text like this is known as a text segment. Start rockin' your translations now!</source>
	      <translation>Un tel extrait est qualifié de segment du texte chez nous. C’est parti – commence tes traductions tout de suite/maintenant !</translation>
    </message>
    <message>
        <source>Arriba, Arriba! Andale, Andale! Be fast as Speedy Gonzales. Just hit TAB to save and go to the next text segment, once you're done.</source>
	      <translation>Arriba, Arriba ! Andale, Andale ! Prends Speedy Gonzales de vitesse. Appuie sur TAB pour sauvegarder et continuer avec le segment prochain dès que tu as fini.</translation>
    </message>
    <message>
        <source>Made with ❤</source>
        <translation>❤</translation>
        <comment>lh-check { min: 10, max: 15 }</comment>
    </message>
</context>
</TS>
'''

## Часть 1

Мы хотим:
- уметь распарсить файлы разного вида
- уметь переводить файлы из одного формата в другой
- для тех форматов, где язык не указан, хотим вызывать какой-то метод/функцию для определения языка
- уметь собирать статистику над файлами всех форматов (что-то типа "а посчитайте мне, сколько раз среди заданного куска массива с файлами переводов какой язык встречается")

In [10]:
def detect_language(text):
    try:
        return detect(text)
    except LangDetectException:
        return "und"


class Format1:
    @staticmethod
    def loads(text):
        root = ET.fromstring(text.strip())
        file = root.find("{*}file")
        entries = {
            unit.get("resname", unit.get("id")): (
                unit.findtext("{*}source", ""),
                unit.findtext("{*}target", ""),
            )
            for unit in file.findall(".//{*}trans-unit")
        }
        return entries, file.get("source-language"), file.get("target-language")

    @staticmethod
    def dumps(data):
        root = ET.Element("xliff", version="1.2", xmlns="urn:oasis:names:tc:xliff:document:1.2")
        file = ET.SubElement(root, "file", {
            "original": "translations", "datatype": "plaintext",
            "source-language": data.source_language, "target-language": data.language,
        })
        body = ET.SubElement(file, "body")
        for index, (key, (source, target)) in enumerate(data.entries.items(), 1):
            unit = ET.SubElement(body, "trans-unit", id=str(index), resname=key)
            ET.SubElement(unit, "source").text = source
            ET.SubElement(unit, "target").text = target
        return ET.tostring(root, encoding="unicode")


class Format2:
    @staticmethod
    def loads(text):
        file = polib.pofile(text)
        entries = {
            str(index): (entry.msgid, entry.msgstr)
            for index, entry in enumerate(file, 1)
            if not entry.obsolete
        }
        return entries, file.metadata.get("X-Source-Language"), file.metadata.get("Language")

    @staticmethod
    def dumps(data):
        file = polib.POFile()
        file.metadata = {
            "Content-Type": "text/plain; charset=UTF-8",
            "X-Source-Language": data.source_language,
            "Language": data.language,
        }
        for source, target in data.entries.values():
            file.append(polib.POEntry(msgid=source, msgstr=target))
        return str(file)


class Format3:
    @staticmethod
    def loads(text):
        root = ET.fromstring(text.strip())
        entries = {
            str(index): (message.findtext("source", ""), message.findtext("translation", ""))
            for index, message in enumerate(root.findall(".//message"), 1)
        }
        return entries, root.get("sourcelanguage"), root.get("language")

    @staticmethod
    def dumps(data):
        root = ET.Element("TS", version="2.1", language=data.language, sourcelanguage=data.source_language)
        context = ET.SubElement(root, "context")
        ET.SubElement(context, "name").text = "translations"
        for source, target in data.entries.values():
            message = ET.SubElement(context, "message")
            ET.SubElement(message, "source").text = source
            ET.SubElement(message, "translation").text = target
        return ET.tostring(root, encoding="unicode")


def stats(ff):
    return dict(Counter(file.language.split("-")[0] for file in ff))

In [11]:
class UniversalFormat:
    def __init__(self, text, format_class, *args):
        self.entries, source_language, language = format_class.loads(text, *args)
        self.source_language = source_language or detect_language(
            " ".join(source for source, target in self.entries.values())
        )
        self.language = language or detect_language(
            " ".join(target for source, target in self.entries.values())
        )
        self.source_language = self.source_language.replace("_", "-")
        self.language = self.language.replace("_", "-")

    def to(self, format_class):
        return format_class.dumps(self)

## Часть 2

Мы нашли еще данных, но уже в других форматах:

In [12]:
# .csv
file6 = '''
"Title","Description","en", "de-AT"
"welcome",,"Hooray, you're here! The day just got better - enjoy the following tips!", "Hurra, du bist hier! Der Tag ist gerettet - nutze die folgenden Tipps!"
"text_segment",,"A section of text like this is known as a text segment. Start rockin' your translations now!", "Eine Textpassage wie diese bezeichnen wir als Textsegment. Starte jetzt mit deinen Übersetzungen durch!"
"tab_shortcut",,"Arriba, Arriba! Andale, Andale! Be fast as Speedy Gonzales. Just hit TAB to save and go to the next text segment, once you're done.", "Arriba, Arriba! Andale, Andale! Sei schneller als Speedy Gonzales. Springe mit TAB ins nächste Textsegment, sobald du fertig bist. Deine Änderungen werden automatisch gespeichert."
"love","lh-check { max: 15, min: 10 }","Made with ❤", "❤"
'''

# .resw
file7_1 = '''
<?xml version="1.0" encoding="utf-8"?>
<root>
  <data name="welcome" xml:space="preserve">
    <value>Hooray, you're here! The day just got better - enjoy the following tips!</value>
  </data>
  <data name="text_segment" xml:space="preserve">
    <value>A section of text like this is known as a text segment. Start rockin' your translations now!</value>
  </data>
  <data name="tab_shortcut" xml:space="preserve">
    <value>Arriba, Arriba! Andale, Andale! Be fast as Speedy Gonzales. Just hit TAB to save and go to the next text segment, once you're done.</value>
  </data>
  <data name="love">
    <value>Made with ❤</value>
    <comment>lh-check { min: 10, max: 15 }</comment>
  </data>
</root>
'''

file7_2 = '''
<?xml version="1.0" encoding="utf-8"?>
<root>
  <data name="welcome" xml:space="preserve">
    <value>Hurrah, tu es là. Ça fait une bonne journée - profite des renseignements suivants !</value>
  </data>
  <data name="text_segment" xml:space="preserve">
    <value>Un tel extrait est qualifié de segment du texte chez nous. C’est parti – commence tes traductions tout de suite/maintenant !</value>
  </data>
  <data name="tab_shortcut" xml:space="preserve">
    <value>Arriba, Arriba ! Andale, Andale ! Prends Speedy Gonzales de vitesse. Appuie sur TAB pour sauvegarder et continuer avec le segment prochain dès que tu as fini.</value>
  </data>
  <data name="love" xml:space="preserve">
    <value>❤</value>
  </data>
</root>
'''

# .php
file8_1 = '''
<?php
  define("welcome", "Hooray, you're here! The day just got better - enjoy the following tips!");

  define("text_segment", "A section of text like this is known as a text segment. Start rockin' your translations now!");

  define("tab_shortcut", "Arriba, Arriba! Andale, Andale! Be fast as Speedy Gonzales. Just hit TAB to save and go to the next text segment, once you're done.");

  # lh-check { min: 10, max: 15 }
  define("love", "Made with ❤");
?>
'''

file8_2 = '''
<?php
  define("welcome", "Hurra, du bist hier! Der Tag ist gerettet - nutze die folgenden Tipps!");

  define("text_segment", "Eine Textpassage wie diese bezeichnen wir als Textsegment. Starte jetzt mit deinen Übersetzungen durch!");

  define("tab_shortcut", "Arriba, Arriba! Andale, Andale! Sei schneller als Speedy Gonzales. Springe mit TAB ins nächste Textsegment, sobald du fertig bist. Deine Änderungen werden automatisch gespeichert.");

  define("love", "❤");
?>
'''

Как именно изменится ваша модель классов для того, чтобы поддержать новые форматы?

Модель не меняется: для каждого нового формата добавляется класс с методами loads и dumps. Ниже пример для CSV. Для RESW и PHP добавляются аналогичные классы, принимающие оригинал и перевод отдельными файлами. UniversalFormat, определение языка и stats остаются прежними.

In [13]:
class Format4:
    @staticmethod
    def loads(text):
        rows = csv.reader(io.StringIO(text.strip()), skipinitialspace=True)
        header = next(rows)
        entries = {row[0]: (row[2], row[3]) for row in rows}
        return entries, header[2], header[3]

    @staticmethod
    def dumps(data):
        output = io.StringIO()
        writer = csv.writer(output)
        writer.writerow(["Title", "Description", data.source_language, data.language])
        writer.writerows(
            [key, "", source, target]
            for key, (source, target) in data.entries.items()
        )
        return output.getvalue()

csv_data = UniversalFormat(file6, Format4)